# Plot picks 

This notebook is used to plot the waveforms within a window.
The window is predefined. 

Agentic AI was used in this notebook.

by Hiroto Bito 

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read
from obspy.core import UTCDateTime
import numpy as np
import os
from tqdm import tqdm

import time

In [4]:
event_df = pd.read_csv('/wd1/hbito_data/data/datasets_all_regions/Cascadia_relocated_catalog_ver_3.csv')
event_df.head()

,Latitude,Longitude,Depth (km),Origin Time (UTC),Uncertainity (km),Horizontal Uncertainity (km),Geometric Std. (km),Detection Value,Num. P,Num. S,RMS Residual (s),Event ID
0,47.22533,-122.16895,56.111,2010-01-01T00:15:17.262282Z,10.223,10.216,0.790,0.680,2,5,1.081,0
1,48.19518,-121.77276,3.820,2010-01-01T00:16:49.375360Z,7.560,3.786,0.140,0.840,25,30,0.985,1
2,47.86208,-122.09903,17.799,2010-01-01T07:18:03.689209Z,5.118,4.807,0.195,0.741,10,18,0.784,2
3,47.96435,-122.91906,21.286,2010-01-01T08:51:56.371091Z,1.899,1.884,0.287,0.756,10,10,0.465,3
4,45.87262,-122.19180,9.822,2010-01-01T16:12:43.838660Z,2.842,2.838,0.229,0.850,20,19,0.657,4


In [6]:
# picks_df = pd.read_csv('/wd1/hbito_data/data/datasets_all_regions/Cascadia_relocated_catalog_picks_ver_3.csv')
# picks_df

,Pick Time (UTC),Station Name,Phase Type,Residual (s),Event ID,Pick ID
0,2010-01-01T00:15:27.180000Z,PCMD.UW,0,0.049,0,0
1,2010-01-01T00:15:37.840400Z,RVW.UW,0,1.264,0,1
2,2010-01-01T00:15:33.280000Z,PCMD.UW,1,-0.243,0,2
3,2010-01-01T00:15:42.002000Z,GNW.UW,1,2.402,0,3
4,2010-01-01T00:15:43.618400Z,B013.PB,1,-0.651,0,4
...,...,...,...,...,...,...
1004330,2015-06-23T23:18:40.885798Z,J11D.7D,1,0.044,63886,1004330
1004331,2015-06-23T23:18:48.573898Z,G35D.7D,1,0.358,63886,1004331
1004332,2015-06-23T23:18:50.458298Z,J19D.7D,1,0.300,63886,1004332
1004333,2015-06-23T23:18:56.689277Z,J10D.7D,1,0.432,63886,1004333


In [10]:
assigned_picks_df = pd.read_csv('/wd1/hbito_data/data/datasets_all_regions/Cascadia_updated_catalog_picks_assignment_ver_3.csv')
assigned_picks_df.head(10)

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3
5,5,2010-01-01T00:15:43.768400Z,PB.B943,S,-0.511,0,5,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813202,-122.911301,84.2
6,6,2010-01-01T00:15:48.060400Z,UW.BOW,S,-0.263,0,6,47.13396,-122.09098,60.147,0.68,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.474831,-123.229301,870.0
7,7,2010-01-01T00:17:04.730000Z,UW.PASS,P,-0.499,1,7,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,48.998299,-122.085197,175.4
8,8,2010-01-01T00:17:05.008400Z,PB.B943,P,-0.252,1,8,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,47.813202,-122.911301,84.2
9,9,2010-01-01T00:17:05.020400Z,UW.BLN,P,0.415,1,9,48.17742,-121.83289,6.163,0.84,2010-01-01 00:16:49.343000+00:00,0.985,25,30,55,48.006624,-122.972646,601.0


In [11]:
assigned_picks_df.keys()

Index(['Unnamed: 0', 'time_pick', 'station', 'phase', 'timeres', 'idx', 'arid',
       'latitude', 'longitude', 'depth', 'Detection Value', 'time',
       'RMS Residual (s)', 'Num. P', 'Num. S', 'picks', 'slatitude',
       'slongitude', 'selevation'],
      dtype='object')

In [ ]:
window_after = 120  # seconds after origin time
window_before = 30  # seconds before origin time

In [17]:
os.makedirs('wd1/hbito_data/data/datasets_all_regions/plot_picks_reloc_cog_ver3/', exist_ok=True)

test

In [21]:
assigned_picks_df.iloc[0]

Unnamed: 0                                         0
time_pick                2010-01-01T00:15:27.180000Z
station                                      UW.PCMD
phase                                              P
timeres                                        0.049
idx                                                0
arid                                               0
latitude                                    47.13396
longitude                                 -122.09098
depth                                         60.147
Detection Value                                 0.68
time                2010-01-01 00:15:16.204000+00:00
RMS Residual (s)                               1.081
Num. P                                             2
Num. S                                             5
picks                                              7
slatitude                                  46.888962
slongitude                               -122.301483
selevation                                    

end test

In [ ]:
# Function to plot waveforms for the first 100 picks using Obspy bulk request
 
def plot_waveforms_for_picks_bulk(assigned_picks_df, event_df, window_before=30, window_after=120, n_picks=100, client_name="IRIS"):
    client = Client(client_name)
    
    # Prepare bulk request list: (network, station, location, channel, starttime, endtime)
    bulk = []
    pick_info = []
    
    for i, row in assigned_picks_df.head(n_picks).iterrows():
        event_id = row['idx']
        pick_id = row['arid']
        origin_time = UTCDateTime(row['time'])
        latitude = row['latitude']
        longitude = row['longitude']
        depth = row['depth']
        
        # You may need to adjust these fields to match your DataFrame columns
        network = row['station'].split('.')[0].strip()
        station = row['station'].split('.')[1].strip()
        location = row.get('location', "*")
        channel = row.get('channel', "BH*")
        starttime = origin_time - window_before
        endtime = origin_time + window_after
        bulk.append((network, station, location, channel, starttime, endtime))
        pick_info.append({
            'event_id': event_id,
            'pick_id': pick_id,
            'origin_time': origin_time,
            'latitude': latitude,
            'longitude': longitude,
            'depth': depth,
        })
    
    # Request all waveforms in bulk (may take time and may fail for some requests)

    time.sleep(0.1)

    try:
        st_list = client.get_waveforms_bulk(bulk)
    except Exception as e:
        print(f"Bulk request failed: {e}")
        st_list = []

    time.sleep(0.1)
    
    # Plotting
    for i, info in enumerate(pick_info):
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.set_title(f"Event ID: {info['event_id']} | Pick ID: {info['pick_id']}")
        ax.set_xlabel(f"Time (s) from Origin Time: {info['origin_time']}")
        ax.set_ylabel("Amplitude")
        
        if i < len(st_list) and st_list[i]:
            st = st_list[i]
            for tr in st:
                times = np.linspace(-window_before, window_after, tr.stats.npts)
                ax.plot(times, tr.data, label=tr.id)
            ax.legend()
        else:
            times = np.linspace(-window_before, window_after, 1000)
            ax.plot(times, np.zeros_like(times), label="No waveform found")
            ax.legend()
        
        # Annotate event info
        info_text = (f"Origin Time: {info['origin_time']}\n"
                     f"Latitude: {info['latitude']}\n"
                     f"Longitude: {info['longitude']}\n"
                     f"Depth: {info['depth']}")
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.98, 0.98, info_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', horizontalalignment='right', bbox=props)
        plt.tight_layout()
        plt.show()
        fig.savefig(f"waveform_plot_{info['event_id']}_{info['pick_id']}.png")
        plt.close(fig)
        time.sleep(0.1)
 
# Run the plotting function (set n_picks=10 for quick test, 100 for full)
plot_waveforms_for_picks_bulk(assigned_picks_df, event_df, window_before, window_after, n_picks=10)